<a href="https://colab.research.google.com/github/chithrakumardakshan-cloud/northstar-analytics/blob/main/notebooks/01_sql_in_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# NORTHSTAR URBAN MOBILITY ANALYTICS
# SQL IN R ANALYSIS
# ==========================================

# STEP 1 — INSTALL AND LOAD PACKAGES

install.packages("sqldf")
install.packages("dplyr")
install.packages("ggplot2")

library(sqldf)
library(dplyr)
library(ggplot2)

cat("Libraries loaded successfully.")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Libraries loaded successfully.

In [2]:
# STEP 2 — LOAD DATASETS

orders <- read.csv("orders.csv")
deliveries <- read.csv("deliveries.csv")
customers <- read.csv("customers.csv")
drivers <- read.csv("drivers.csv")
vehicles <- read.csv("vehicles.csv")
hubs <- read.csv("hubs.csv")
incidents <- read.csv("incidents.csv")
complaints <- read.csv("complaints.csv")
app_events <- read.csv("app_events.csv")

cat("Datasets loaded successfully.")

Datasets loaded successfully.

In [3]:
# STEP 3 — PREVIEW DATA

head(orders)

head(deliveries)

str(orders)

str(deliveries)

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>
1,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
2,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
3,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
4,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
5,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0
6,O00006,C0437,Retail,2024-08-05 04:55:00,1,CENTRAL,East,High,151.44,Web,1


,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>
1,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05
2,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41
3,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51
4,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62
5,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22
6,DL00006,O00029,D037,V098,H03,2024-09-11 12:40:00,2024-09-12 17:11:52.384869,Delayed,13.84,0,0,1.57,9.58


'data.frame':	1250 obs. of  11 variables:
 $ order_id             : chr  "O00001" "O00002" "O00003" "O00004" ...
 $ customer_id          : chr  "C0292" "C0459" "C0161" "C0520" ...
 $ service_type         : chr  "Passenger" "Passenger" "Passenger" "Parcel" ...
 $ order_created_at     : chr  "2024-08-20 14:43:00" "2024-05-14 22:16:00" "2025-09-02 14:37:00" "2025-01-11 17:15:00" ...
 $ promised_window_hours: int  6 24 4 2 12 1 2 4 12 6 ...
 $ pickup_zone          : chr  "Airport" "North" "West" "RiverSide" ...
 $ dropoff_zone         : chr  "South" "AIRPORT" "AIRPORT" "North" ...
 $ priority_level       : chr  "Medium" "Low" "High" "Medium" ...
 $ order_value          : num  126.7 109.3 33.5 10 125.6 ...
 $ booking_channel      : chr  "App" "App" "Phone" "App" ...
 $ special_handling_flag: int  0 0 0 1 0 1 0 0 0 0 ...
'data.frame':	950 obs. of  13 variables:
 $ delivery_id                  : chr  "DL00001" "DL00002" "DL00003" "DL00004" ...
 $ order_id                     : chr  "O00938" "

In [4]:
# QUERY 1 — DELIVERY FAILURE RATE BY HUB

query1 <- sqldf("
SELECT
    h.hub_name,
    h.zone,
    COUNT(d.delivery_id) AS total_deliveries,

    SUM(
        CASE
            WHEN d.delivery_status = 'Failed'
            THEN 1
            ELSE 0
        END
    ) AS failed_deliveries,

    ROUND(
        SUM(
            CASE
                WHEN d.delivery_status = 'Failed'
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(d.delivery_id),
        2
    ) AS failure_rate

FROM deliveries d

JOIN hubs h
ON d.hub_id = h.hub_id

GROUP BY h.hub_name, h.zone

ORDER BY failure_rate DESC
")

query1

hub_name,zone,total_deliveries,failed_deliveries,failure_rate
<chr>,<chr>,<int>,<int>,<dbl>
Midtown Relay,Central,128,26,20.31
Central Core,Central,115,23,20.00
Airport Hub,Airport,104,15,14.42
West Gate,West,127,16,12.60
North Exchange,North,136,17,12.50
Riverside Hub,Riverside,115,14,12.17
South Link,South,106,10,9.43
East Dock,East,119,11,9.24


In [5]:
# QUERY 2 — DRIVER PERFORMANCE ANALYSIS

query2 <- sqldf("
SELECT
    d.driver_id,
    dr.base_zone,
    dr.employment_type,

    COUNT(d.delivery_id) AS total_deliveries,

    ROUND(
        AVG(d.manual_route_override_count),
        2
    ) AS avg_overrides,

    ROUND(
        AVG(d.customer_rating_post_delivery),
        2
    ) AS avg_customer_rating

FROM deliveries d

JOIN drivers dr
ON d.driver_id = dr.driver_id

GROUP BY
    d.driver_id,
    dr.base_zone,
    dr.employment_type

HAVING total_deliveries >= 3

ORDER BY avg_overrides DESC
")

query2

driver_id,base_zone,employment_type,total_deliveries,avg_overrides,avg_customer_rating
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>
D127,CENTRAL,FullTime,6,2.83,4.10
D062,South,FullTime,3,2.00,3.82
D069,NORTH,PartTime,7,2.00,3.94
D085,North,PartTime,4,2.00,3.42
D105,RiverSide,Contract,7,2.00,4.21
D124,north,FullTime,4,2.00,3.41
D130,WEST,FullTime,8,2.00,3.80
D139,South,FullTime,5,2.00,4.08
D028,North,FullTime,7,1.86,3.54


In [6]:
# QUERY 3 — CUSTOMER COMPLAINT ANALYSIS

query3 <- sqldf("
SELECT
    c.customer_id,
    c.home_zone,
    c.customer_type,

    COUNT(cp.complaint_id) AS total_complaints,

    SUM(
        CASE
            WHEN d.delivery_status = 'Failed'
            THEN 1
            ELSE 0
        END
    ) AS failed_deliveries,

    ROUND(
        SUM(cp.compensation_amount),
        2
    ) AS total_compensation

FROM customers c

LEFT JOIN complaints cp
ON c.customer_id = cp.customer_id

LEFT JOIN orders o
ON c.customer_id = o.customer_id

LEFT JOIN deliveries d
ON o.order_id = d.order_id

GROUP BY
    c.customer_id,
    c.home_zone,
    c.customer_type

HAVING total_complaints >= 2

ORDER BY total_complaints DESC
")

query3

customer_id,home_zone,customer_type,total_complaints,failed_deliveries,total_compensation
<chr>,<chr>,<chr>,<int>,<int>,<dbl>
C0372,West,Consumer,18,3,431.34
C0545,SOUTH,Consumer,18,0,441.60
C0242,East,Consumer,15,0,378.75
C0023,South,Consumer,12,2,265.44
C0172,north,Consumer,12,0,245.96
C0335,NORTH,SME,12,0,277.32
C0368,North,Consumer,12,4,232.53
C0599,North,Consumer,12,0,52.44
C0622,RiverSide,Consumer,12,0,109.14


In [7]:
# QUERY 4 — VEHICLE BATTERY HEALTH ANALYSIS

query4 <- sqldf("
SELECT

    CASE
        WHEN v.battery_health_pct >= 80
        THEN 'Excellent'

        WHEN v.battery_health_pct >= 60
        THEN 'Moderate'

        ELSE 'Poor'
    END AS battery_category,

    COUNT(DISTINCT v.vehicle_id) AS total_vehicles,

    COUNT(i.incident_id) AS incident_count,

    ROUND(
        AVG(i.resolved_hours),
        2
    ) AS avg_resolution_time

FROM vehicles v

JOIN deliveries d
ON v.vehicle_id = d.vehicle_id

LEFT JOIN incidents i
ON d.delivery_id = i.delivery_id

GROUP BY battery_category
")

query4

battery_category,total_vehicles,incident_count,avg_resolution_time
<chr>,<int>,<int>,<dbl>
Excellent,47,115,11.41
Moderate,56,121,12.82
Poor,17,44,11.31


In [8]:
# QUERY 5 — SERVICE PROFITABILITY ANALYSIS

query5 <- sqldf("
SELECT
    o.pickup_zone,
    o.service_type,

    COUNT(o.order_id) AS total_orders,

    ROUND(
        AVG(o.order_value),
        2
    ) AS avg_order_value,

    ROUND(
        AVG(d.fuel_or_charge_cost),
        2
    ) AS avg_delivery_cost,

    ROUND(
        AVG(o.order_value - d.fuel_or_charge_cost),
        2
    ) AS avg_profit_margin

FROM orders o

JOIN deliveries d
ON o.order_id = d.order_id

GROUP BY
    o.pickup_zone,
    o.service_type

ORDER BY avg_profit_margin ASC
")

query5

pickup_zone,service_type,total_orders,avg_order_value,avg_delivery_cost,avg_profit_margin
<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>
West,Business,7,53.95,13.15,40.81
East,Medical,9,52.86,11.89,40.97
Ctr,Medical,2,53.92,11.13,42.79
north,Medical,7,57.49,14.17,43.32
AIRPORT,Business,5,66.61,18.22,48.39
Central,Passenger,20,61.75,11.90,49.85
RiverSide,Parcel,17,68.07,13.28,54.80
North,Parcel,8,67.15,10.84,56.31
East,Retail,20,69.57,11.73,57.84


In [9]:
# STEP 4 — QUERY OPTIMISATION

# BEFORE OPTIMISATION

slow_query <- sqldf("
SELECT *
FROM deliveries d

JOIN orders o
ON d.order_id = o.order_id

WHERE d.delivery_status = 'Failed'
")

# AFTER OPTIMISATION

failed_deliveries <- deliveries[
    deliveries$delivery_status == 'Failed',
]

fast_query <- sqldf("
SELECT *
FROM failed_deliveries f

JOIN orders o
ON f.order_id = o.order_id
")

cat("Original delivery rows:", nrow(deliveries), "\n")

cat("Filtered failed rows:", nrow(failed_deliveries), "\n")

Original delivery rows: 950 
Filtered failed rows: 132 
